# Hedging the guarantee: results in reading orderThis notebook reads what the scripts wrote. It does not recompute anything, so it runs in acouple of seconds and it cannot disagree with the tables in `reports/tables/`. Run`make all` first, or at least `make data valuation greeks validate backtest accounting`.Order: what the contract is worth, what its risk looks like, whether the model matchesJackson's disclosure, what hedging does, and why reported earnings still move.

In [ ]:
from pathlib import Pathimport pandas as pdfrom IPython.display import Image, displaypd.set_option("display.width", 200)pd.set_option("display.max_columns", 50)ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()TABLES = ROOT / "reports" / "tables"FIGURES = ROOT / "reports" / "figures"def table(name):    return pd.read_csv(TABLES / f"{name}.csv")def figure(name):    display(Image(filename=str(FIGURES / f"{name}.png")))sorted(p.stem for p in TABLES.glob("*.csv"))

## 1. The contract at issueA $100,000 single premium at issue age 70, Perspective II with the Flex GMWB Core rider, valuedat 31 December 2025.The line to look at is the market risk benefit. It comes out at zero, and that is a checkrather than a coincidence: Jackson's Note 6 says the benefit starts at a fair value of zerowhere projected attributed fees cover projected claims, and the attribution percentage thatcalibration produces is 0.87, so they do.

In [ ]:
table("valuation_headline")

Why claims reach 19% of premium on a risk-neutral basis: a 5.75% lifetime draw against arisk-neutral drift near 3.7% less 2.26% of charges depletes the account on most paths. Firstclaims arrive around year five and by year twenty the account is gone on 86% of paths. Theguarantee is the expected outcome for this contract, not a tail event.

In [ ]:
figure("cash_flow_profile")

In [ ]:
flows = table("cash_flow_profile")flows.loc[flows["policy_year"] <= 25, ["policy_year", "expected_claim", "expected_rider_charge",                                       "expected_base_charge", "prob_exhausted",                                       "mean_account_value", "mean_benefit_base"]].round(1)

## 2. How much of this is assumptionOne-at-a-time variations, each repriced on the same random draws. The rate shock and thestep-up move the answer most; the projection-truncation rows at the bottom are there to showthat the horizon cap costs an order of magnitude less than Monte Carlo error.

In [ ]:
table("valuation_robustness").round(1)

In [ ]:
table("monte_carlo_convergence").round(2)

## 3. RiskEquity exposure is negative and large: a quarter of the account value has to be sold short toneutralise it. Rho is negative because a long-dated liability discounts away faster when ratesrise. Most of the volatility exposure sits in the long-run level rather than in the part a deskcan trade.

In [ ]:
display(table("greeks").round(2))display(table("hedge_sizing").round(2))

The volatility term structure is fitted to the 3-month implied index and mean reverts to along-run realised level. The 30-day index is plotted as an observation the curve was **not**fitted to.

In [ ]:
figure("volatility_term_structure")

Value and sensitivity across moneyness. At issue the benefit base equals the account value; abook that has ridden markets up sits to the left of that.

In [ ]:
figure("moneyness_curve")

### Where equity exposure changes signNot every state leaves the insurer short the index. The benefit base resets to the contractvalue on the anniversary, so with the contract value above the benefit base the index level onthat one date fixes the guaranteed income for life, and both the claims leg and the fees legrespond positively to the index. The net is their difference, which is why the sign isdelicate and why the rate environment decides it.

In [ ]:
table("step_up_proximity").round(1)

## 4. Against the disclosureJackson runs a 10% equity shock and a parallel rate shift every year in Item 7A. Everythingbelow is scaled by variable annuity account value, which is the only way one $100,000 policyand a $236bn book go on the same axis.Sign first.

In [ ]:
table("sign_agreement")

Then shape. Where a filing discloses both 50bp and 100bp, the ratio of the two is a convexitytest the model was never fitted to.

In [ ]:
display(table("convexity_disclosed").round(4))display(table("convexity_model").round(4))

Then scale, on the vintage portfolio: five policies issued between 2016 and 2024 with issueages chosen so the weighted attained age straddles Jackson's disclosed 70.

In [ ]:
portfolio = table("portfolio_vs_disclosed")display(portfolio[["as_of", "vintages", "weighted_attained_age", "weighted_gawa_pct",                   "gwb_over_av", "value_pct_av", "disclosed_value_pct_av"]].round(3))summary = table("portfolio_comparison_summary")display(summary.round(3))print(f"signs agreeing: {int(summary['sign_agrees'].sum())} of {len(summary)}")print(f"ratio of model to disclosed: median {summary['ratio'].median():.2f}, "      f"range {summary['ratio'].min():.2f} to {summary['ratio'].max():.2f}")

The ratio sits near 2.4 across six different shocks and four dates. A near-constant multiplepoints at one structural assumption rather than a pile of errors, and the candidate is thestatic full-utilisation, no-lapse base case. Sweeping both, after scaling for the quarter ofaccount value that carries no withdrawal guarantee:

In [ ]:
behaviour = table("behaviour_reconciliation")behaviour[["utilisation", "lapse_rate", "value_pct_av", "ratio_equity_down_10pct",           "ratio_equity_up_10pct", "ratio_rates_up_100bp", "ratio_rates_down_100bp",           "mean_ratio"]].round(3)

In [ ]:
figure("shock_comparison")

Independently, the guaranteed withdrawal rate on its own moves the rate sensitivity across thewhole range between the model and the disclosure.

In [ ]:
table("withdrawal_rate_diagnostic").round(4)

## 5. HedgingOne policy issued September 2016 on Jackson's disclosed fund mix, hedged weekly for 521 weeks.Each leg is sized from its own Greek, so one run gives every combination.

In [ ]:
display(table("hedge_effectiveness_book_mix").round(2))figure("leg_comparison")

In [ ]:
figure("hedge_performance")

Delta and rho take out about two thirds of the variance and leave the volatility exposurestanding. The stress windows say it more plainly than the summary statistics: the seven weeksto the end of March 2020 cost 30.6% of premium unhedged, and a futures-and-swaps programmerecovered 40% of that.

In [ ]:
display(table("hedge_stress_windows").round(1))display(table("hedge_residual_attribution").round(3))

In [ ]:
table("hedge_exposures").round(1)

Read the variance reduction as an upper bound. The liability is revalued with the same modelthat produced the hedge ratios, so anything the model represents and the hedge covers isremoved nearly completely. What this measures is the cost of hedging discretely with imperfectinstruments, not model error. `docs/limitations.md` has the rest.## 6. Economic against reportedSame hedge positions, measured on the economic basis and on the reporting basis. The reportingbasis carries the annuity table's margins and an own-credit adjustment the economic basis doesnot.

In [ ]:
display(table("accounting_volatility").round(2))display(table("accounting_by_year").round(2))

In [ ]:
figure("economic_versus_reported")

Reported net income is about 5% more volatile than the economic outcome under identicalpositions, in every calendar year of the sample. The more interesting row is reportedcomprehensive income, which is *less* volatile than either: credit spreads widen when equitymarkets fall, so the own-credit adjustment offsets part of the market move. The market riskbenefit rules report that piece in other comprehensive income, so net income gets the undampedmove and comprehensive income gets the damped one. An accounting boundary turns a naturaloffset into a reporting mismatch, and no hedge in this programme touches the piece that sitsoutside net income.

In [ ]:
display(table("accounting_worst_periods").round(1))display(table("accounting_basis_gap").describe().loc[["mean", "50%", "min", "max"]].round(1))